
# 🌍 AI MULTILINGUAL COMPLAINT CHATBOT WITH CACHE SYSTEM + GRADIO UI



In [1]:

# STEP 1 — INSTALL REQUIRED LIBRARIES

!pip install -q pandas numpy sentence-transformers \
scikit-learn deep-translator langdetect gradio


In [2]:
# STEP 2 — IMPORT LIBRARIES

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from deep_translator import GoogleTranslator

from langdetect import detect

import gradio as gr

In [3]:
# STEP 3 — LOAD DATASET

# Dataset columns required:
# complaint
# response
# product
# issue

df = pd.read_csv(
    "/content/sampled_cleaned_complaints.csv"
)

print("✅ Dataset Loaded Successfully!")

print(df.head())

✅ Dataset Loaded Successfully!
                                             product  \
0                                       Student loan   
1                        Checking or savings account   
2                                    Debt collection   
3  Credit reporting or other personal consumer re...   
4  Credit reporting or other personal consumer re...   

                                               issue  \
0                                Can't repay my loan   
1                                 Closing an account   
2  Took or threatened to take negative or legal a...   
3               Incorrect information on your report   
4               Incorrect information on your report   

                                           complaint                 response  
0  MY COMPLAINT IS WITH NAVIENT. NAVIENT FORCED M...  Closed with explanation  
1  I am writing to formally lodge a complaint aga...  Closed with explanation  
2  Have been paying on this car since my son was ...  C

In [4]:
# STEP 4 — DATA CLEANING

df = df.dropna()

df["complaint"] = df["complaint"].astype(str)

df["response"] = df["response"].astype(str)

df["product"] = df["product"].astype(str)

df["issue"] = df["issue"].astype(str)

# Remove very short complaints
df = df[
    df["complaint"].str.len() > 20
]

# Remove very long complaints
df = df[
    df["complaint"].str.len() < 500
]

# Reset index
df = df.reset_index(drop=True)

print("\n✅ Dataset Cleaned!")

print("Total Rows:", len(df))



✅ Dataset Cleaned!
Total Rows: 1815


In [5]:
# STEP 5 — LOAD EMBEDDING MODEL

print("\nLoading Embedding Model...")

embedding_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

print("✅ Embedding Model Loaded!")



Loading Embedding Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding Model Loaded!


In [6]:
# STEP 6 — CREATE EMBEDDINGS


print("\nCreating Embeddings...")

complaints = df["complaint"].tolist()

embeddings = embedding_model.encode(
    complaints,
    show_progress_bar=True
)

print("✅ Embeddings Created Successfully!")



# STEP 7 — LOAD TRANSLATOR

print("✅ Translator Loaded!")


Creating Embeddings...


Batches:   0%|          | 0/57 [00:00<?, ?it/s]

✅ Embeddings Created Successfully!
✅ Translator Loaded!


In [7]:
# STEP 7.5 — CACHE SYSTEM

response_cache = {}

print("✅ Cache System Initialized!")


✅ Cache System Initialized!


In [10]:
# STEP 8 — CHATBOT FUNCTION

def multilingual_chatbot(user_input):

    try:

        # ==================================================
        # EMPTY INPUT CHECK
        # ==================================================

        if user_input.strip() == "":

            return "⚠ Please enter a complaint."

        # ==================================================
        # CACHE CHECK
        # ==================================================

        cache_key = user_input.lower().strip()

        if cache_key in response_cache:

            print("⚡ Response Retrieved From Cache")

            cached_output = response_cache[
                cache_key
            ]

            cached_output += """

# ==================================================
# ⚡ CACHE STATUS
# ==================================================

This response was retrieved from cache.
"""

            return cached_output

        # ==================================================
        # DETECT LANGUAGE
        # ==================================================

        detected_lang = detect(user_input)

        print("Detected Language:", detected_lang)

        # ==================================================
        # TRANSLATE INPUT TO ENGLISH
        # ==================================================

        if detected_lang != "en":

            english_input = GoogleTranslator(
                source='auto',
                target='en'
            ).translate(user_input)

        else:

            english_input = user_input

        print("English Input:", english_input)

        # ==================================================
        # CREATE USER EMBEDDING
        # ==================================================

        user_embedding = embedding_model.encode(
            [english_input]
        )

        # ==================================================
        # COSINE SIMILARITY
        # ==================================================

        similarities = cosine_similarity(
            user_embedding,
            embeddings
        )

        # ==================================================
        # GET BEST MATCH
        # ==================================================

        best_match_idx = similarities.argmax()

        similarity_score = similarities[
            0
        ][best_match_idx]

        # ==================================================
        # RETRIEVE MATCHED DATA
        # ==================================================

        matched_product = df.iloc[
            best_match_idx
        ]["product"]

        matched_issue = df.iloc[
            best_match_idx
        ]["issue"]

        matched_response = df.iloc[
            best_match_idx
        ]["response"]

        # ==================================================
        # GENERATE ENGLISH RESPONSE
        # ==================================================

        english_response = f"""
Issue Category: {matched_product}

Detected Issue: {matched_issue}

Suggested Resolution:

We understand your complaint regarding
{matched_product.lower()}.

Please contact customer support with:
- transaction details
- account information
- proof of payment

Your complaint should be reviewed
and resolved after verification.

Reference Resolution:
{matched_response}

Similarity Score:
{similarity_score:.2f}
"""

        # ==================================================
        # TRANSLATE RESPONSE
        # ==================================================

        if detected_lang != "en":

            try:

                short_response = f"""
Issue Category: {matched_product}

Detected Issue: {matched_issue}

Suggested Resolution:
Please contact customer support with
transaction details and proof of payment.
"""

                translated_response = GoogleTranslator(
                    source='en',
                    target=detected_lang
                ).translate(short_response)

            except Exception as translation_error:

                translated_response = (
                    "Translation Failed: "
                    + str(translation_error)
                )

        else:

            translated_response = english_response

        # ==================================================
        # FINAL OUTPUT
        # ==================================================

        final_output = f"""
==================================================
🌍 ENGLISH RESPONSE
==================================================

{english_response}

--------------------------------------------------

🌐 ORIGINAL LANGUAGE RESPONSE
==================================================

{translated_response}
"""

        # ==================================================
        # STORE RESPONSE IN CACHE
        # ==================================================

        final_output += """

==================================================
🆕 CACHE STATUS
==================================================

This response was newly generated
and stored in cache.
"""

        response_cache[
            cache_key
        ] = final_output

        print("✅ Response Stored In Cache")

        print(
            "Current Cache Size:",
            len(response_cache)
        )

        return final_output

    except Exception as e:

        return f"❌ Error: {str(e)}"


In [11]:

# STEP 8 — CHATBOT FUNCTION

def multilingual_chatbot(user_input):

    try:

        # ==================================================
        # EMPTY INPUT CHECK
        # ==================================================

        if user_input.strip() == "":

            return "⚠ Please enter a complaint."

        # ==================================================
        # CACHE CHECK
        # ==================================================

        cache_key = user_input.lower().strip()

        if cache_key in response_cache:

            print("⚡ Response Retrieved From Cache")

            cached_output = response_cache[
                cache_key
            ]

            cached_output += """

# ==================================================
# ⚡ CACHE STATUS
# ==================================================

This response was retrieved from cache.
"""

            return cached_output

        # ==================================================
        # DETECT LANGUAGE
        # ==================================================

        detected_lang = detect(user_input)

        print("Detected Language:", detected_lang)

        # ==================================================
        # TRANSLATE INPUT TO ENGLISH
        # ==================================================

        if detected_lang != "en":

            english_input = GoogleTranslator(
                source='auto',
                target='en'
            ).translate(user_input)

        else:

            english_input = user_input

        print("English Input:", english_input)

        # ==================================================
        # CREATE USER EMBEDDING
        # ==================================================

        user_embedding = embedding_model.encode(
            [english_input]
        )

        # ==================================================
        # COSINE SIMILARITY
        # ==================================================

        similarities = cosine_similarity(
            user_embedding,
            embeddings
        )

        # ==================================================
        # GET BEST MATCH
        # ==================================================

        best_match_idx = similarities.argmax()

        similarity_score = similarities[
            0
        ][best_match_idx]

        # ==================================================
        # RETRIEVE MATCHED DATA
        # ==================================================

        matched_product = df.iloc[
            best_match_idx
        ]["product"]

        matched_issue = df.iloc[
            best_match_idx
        ]["issue"]

        matched_response = df.iloc[
            best_match_idx
        ]["response"]

        # ==================================================
        # GENERATE ENGLISH RESPONSE
        # ==================================================

        english_response = f"""
Issue Category: {matched_product}

Detected Issue: {matched_issue}

Suggested Resolution:

We understand your complaint regarding
{matched_product.lower()}.

Please contact customer support with:
- transaction details
- account information
- proof of payment

Your complaint should be reviewed
and resolved after verification.

Reference Resolution:
{matched_response}

Similarity Score:
{similarity_score:.2f}
"""

        # ==================================================
        # TRANSLATE RESPONSE
        # ==================================================

        if detected_lang != "en":

            try:

                short_response = f"""
Issue Category: {matched_product}

Detected Issue: {matched_issue}

Suggested Resolution:
Please contact customer support with
transaction details and proof of payment.
"""

                translated_response = GoogleTranslator(
                    source='en',
                    target=detected_lang
                ).translate(short_response)

            except Exception as translation_error:

                translated_response = (
                    "Translation Failed: "
                    + str(translation_error)
                )

        else:

            translated_response = english_response

        # ==================================================
        # FINAL OUTPUT
        # ==================================================

        final_output = f"""
==================================================
🌍 ENGLISH RESPONSE
==================================================

{english_response}

--------------------------------------------------

🌐 ORIGINAL LANGUAGE RESPONSE
==================================================

{translated_response}
"""

        # ==================================================
        # STORE RESPONSE IN CACHE
        # ==================================================

        final_output += """

==================================================
🆕 CACHE STATUS
==================================================

This response was newly generated
and stored in cache.
"""

        response_cache[
            cache_key
        ] = final_output

        print("✅ Response Stored In Cache")

        print(
            "Current Cache Size:",
            len(response_cache)
        )

        return final_output

    except Exception as e:

        return f"❌ Error: {str(e)}"







In [12]:
# STEP 9 — CREATE GRADIO UI

interface = gr.Interface(

    fn=multilingual_chatbot,

    inputs=gr.Textbox(
        lines=6,
        placeholder=
        "Enter your complaint in any language..."
    ),

    outputs=gr.Textbox(
        lines=30,
        label="Chatbot Response"
    ),

    title="🌍 AI Multilingual Complaint Chatbot",

    description="""
This chatbot supports:

✅ Multilingual Complaint Analysis
✅ Sentence Embeddings
✅ Semantic Similarity Search
✅ AI-based Complaint Retrieval
✅ Cache System
✅ English + Original Language Responses
""",

    theme="soft",

    flagging_mode="never"
)

In [13]:
# STEP 10 — LAUNCH APPLICATION


interface.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f155ac3df26e1d0421.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Detected Language: ta
English Input: "Your double transaction problem has been detected."
✅ Response Stored In Cache
Current Cache Size: 1
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f155ac3df26e1d0421.gradio.live
